In [ ]:
# Species temperature fit in a bounded region

findex = 250
chosen_step = 438

In [ ]:
# Imports and Data Loading

from graphet import Data
from graphet.plugins import TristanV2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.patches import Polygon
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun

%matplotlib widget

%load_ext autoreload
%autoreload 2

set_standard_plot_params()

run = TristanRun(findex)
d = run.config_data([chosen_step], "both")

In [ ]:
# Region of interest: trapezoid bounding the inflow/exhaust region

trapezoid = [(530, 1143), (396, 1136), (437, 1562), (498, 1562)]

def choose_quad(df, points):
    path = Path(points)
    points_array = df[['x', 'y']].values
    mask = path.contains_points(points_array)
    return df[mask]

In [ ]:
# Sanity check: show the chosen region on the density map

roi = fq_cut(d.fields, 0, [0, run.xlim, 0, run.ylim])

fig, ax = plt.subplots(figsize=(6, 6))
mesh = plot_fq3(roi, roi.ndens, plot=ax, vabsmax=4)
ax.add_patch(Polygon(trapezoid, closed=True, fill=False, edgecolor='red', linewidth=2))
ax.set_title(f"Selected region, findex={findex}, step={chosen_step}")
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label("$n / n_0$")
plt.show()

In [ ]:
# Select electron and ion particles within the region

species_names = {1: "electron", 2: "ion"}

dfs_full = {idx: d.particles[idx].to_dataframe() for idx in species_names}

for df in dfs_full.values():
    add_gamma(df)

region_dfs = {idx: choose_quad(df, trapezoid) for idx, df in dfs_full.items()}

for idx, df in region_dfs.items():
    print(f"{species_names[idx]}: {len(df)} particles in region, max(gamma - 1) = {(df['gamma'] - 1).max():.2e}")

In [ ]:
# Fit a (non-relativistic, drifting) Maxwellian to each velocity component
# f(v) ~ exp(-(v - mu)^2 / (2 sigma^2)); weighted MLE gives mu, sigma directly as the
# weighted mean and weighted standard deviation of each component

def weighted_mean_std(values, weights):
    mu = np.sum(weights * values) / np.sum(weights)
    var = np.sum(weights * (values - mu) ** 2) / np.sum(weights)
    return mu, np.sqrt(var)

def fit_drifting_maxwellian(df):
    weights = df['wei'].to_numpy(dtype=float)
    fit = {}
    for component in ['u', 'v', 'w']:
        values = df[component].to_numpy(dtype=float)
        fit[component] = weighted_mean_std(values, weights)
    return fit

fits = {idx: fit_drifting_maxwellian(df) for idx, df in region_dfs.items()}
fits

In [ ]:
# Plot each species' velocity distributions with the Maxwellian fits overlaid

def gaussian(v, mu, sigma):
    return np.exp(-(v - mu) ** 2 / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))

fig, axes = plt.subplots(2, 3, figsize=(11, 5), constrained_layout=True)

for row, idx in enumerate(species_names):
    df = region_dfs[idx]
    weights = df['wei'].to_numpy(dtype=float)

    for col, component in enumerate(['u', 'v', 'w']):
        ax = axes[row, col]
        values = df[component].to_numpy(dtype=float)

        hist, bin_edges = np.histogram(values, bins=60, weights=weights, density=True)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        mu, sigma = fits[idx][component]
        v_fit = np.linspace(bin_edges[0], bin_edges[-1], 300)

        ax.bar(bin_centers, hist, width=np.diff(bin_edges), color='steelblue', alpha=0.6, label='simulation')
        ax.plot(v_fit, gaussian(v_fit, mu, sigma), 'r--', lw=2, label=fr'fit: $\mu$={mu:.2e}, $\sigma$={sigma:.2e}')

        ax.set_xlabel(f'${component}$')
        if col == 0:
            ax.set_ylabel('density')
        ax.set_title(f'{species_names[idx]}: ${component}$')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

fig.suptitle(f"Drifting Maxwellian fits, findex={findex}, step={chosen_step}")
plt.show()

In [ ]:
# Temperature from the thermal spread: for a non-relativistic Maxwellian the variance of a
# single velocity component is sigma^2 = T / (m c^2); electrons have mass 1 (code units),
# ions have mass run.mime

masses = {1: 1.0, 2: run.mime}

temperatures = {}
for idx in species_names:
    sigmas = np.array([fits[idx][c][1] for c in ['u', 'v', 'w']])
    theta = np.mean(sigmas ** 2)  # T / (m c^2), isotropic average over the 3 components
    temperatures[idx] = masses[idx] * theta  # T / (m_e c^2)
    print(f"{species_names[idx]}: sigma_(u,v,w) = {sigmas}, T/(m c^2) = {theta:.4e}, T/(m_e c^2) = {temperatures[idx]:.4e}")

T_ratio = temperatures[2] / temperatures[1]
print(f"\nT_i / T_e = {T_ratio:.3f}")

In [ ]:
# Naive energy partition: ignore the Maxwellian fit entirely and just compare the total
# kinetic energy carried by each species in the region (includes bulk flow, doesn't account
# for differing particle counts/degrees of freedom -- a sanity check on how much the fit matters)

total_energy = {}
for idx in species_names:
    df = region_dfs[idx]
    weights = df['wei'].to_numpy(dtype=float)
    speed_sq = (
        df['u'].to_numpy(dtype=float) ** 2
        + df['v'].to_numpy(dtype=float) ** 2
        + df['w'].to_numpy(dtype=float) ** 2
    )
    total_energy[idx] = np.sum(weights * 0.5 * masses[idx] * speed_sq)
    print(f"{species_names[idx]}: total KE = {total_energy[idx]:.4e}")

T_ratio_naive = total_energy[2] / total_energy[1]
print(f"\nNaive T_i / T_e (total ion energy / total electron energy) = {T_ratio_naive:.3f}")

In [ ]:
# Bulk flow (drift) of each species, and the KE ratio if each species' flow were "cold"
# (i.e. every particle exactly at the bulk velocity, with no thermal spread at all)

bulk_velocity = {}
bulk_speed_sq = {}
N_total = {}
KE_bulk = {}

for idx in species_names:
    mu_u, _ = fits[idx]['u']
    mu_v, _ = fits[idx]['v']
    mu_w, _ = fits[idx]['w']
    bulk_velocity[idx] = (mu_u, mu_v, mu_w)
    bulk_speed_sq[idx] = mu_u ** 2 + mu_v ** 2 + mu_w ** 2

    weights = region_dfs[idx]['wei'].to_numpy(dtype=float)
    N_total[idx] = np.sum(weights)
    KE_bulk[idx] = N_total[idx] * 0.5 * masses[idx] * bulk_speed_sq[idx]

    print(f"{species_names[idx]}: bulk (u, v, w) = ({mu_u:.4e}, {mu_v:.4e}, {mu_w:.4e}), "
          f"|v_bulk| = {np.sqrt(bulk_speed_sq[idx]):.4e}")

KE_bulk_ratio = KE_bulk[2] / KE_bulk[1]
print(f"\nCold-flow KE ratio (ion bulk KE / electron bulk KE) = {KE_bulk_ratio:.3f}")

In [ ]:
# Naive energy partition in the flow frame of each species: subtract each species' own bulk
# drift before summing kinetic energy, to isolate how much of the raw (lab-frame) energy ratio
# was just due to differing bulk flow speeds (this still folds in any difference in particle
# counts/weights between species, unlike the per-particle Maxwellian-fit temperature)

thermal_energy = {}
for idx in species_names:
    df = region_dfs[idx]
    weights = df['wei'].to_numpy(dtype=float)
    mu_u, mu_v, mu_w = bulk_velocity[idx]

    du = df['u'].to_numpy(dtype=float) - mu_u
    dv = df['v'].to_numpy(dtype=float) - mu_v
    dw = df['w'].to_numpy(dtype=float) - mu_w

    thermal_energy[idx] = np.sum(weights * 0.5 * masses[idx] * (du ** 2 + dv ** 2 + dw ** 2))
    print(f"{species_names[idx]}: thermal (flow-frame) KE = {thermal_energy[idx]:.4e}")

T_ratio_flow_frame = thermal_energy[2] / thermal_energy[1]
print(f"\nNaive T_i / T_e in each species' flow frame = {T_ratio_flow_frame:.3f}")

In [ ]:
# T_i / T_e in the region: Maxwellian fit vs. naive lab-frame energy vs. naive flow-frame energy

fig, axes = plt.subplots(1, 3, figsize=(13, 5))

axes[0].bar(['electron', 'ion'], [temperatures[1], temperatures[2]], color=['steelblue', 'indianred'])
axes[0].set_ylabel('$T / (m_e c^2)$')
axes[0].set_title(fr'Maxwellian fit: $T_i/T_e$ = {T_ratio:.2f}')
axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(['electron', 'ion'], [total_energy[1], total_energy[2]], color=['steelblue', 'indianred'])
axes[1].set_ylabel('total kinetic energy')
axes[1].set_title(fr'Naive (lab frame): $T_i/T_e$ = {T_ratio_naive:.2f}')
axes[1].grid(alpha=0.3, axis='y')

axes[2].bar(['electron', 'ion'], [thermal_energy[1], thermal_energy[2]], color=['steelblue', 'indianred'])
axes[2].set_ylabel('total kinetic energy (flow frame)')
axes[2].set_title(fr'Naive (flow frame): $T_i/T_e$ = {T_ratio_flow_frame:.2f}')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Decompose each particle's velocity into components parallel and perpendicular to the local B
# (using each particle's own interpolated bx, by, bz), to allow separate T_par / T_perp fits

def perpendicular_basis(bx, by, bz):
    Bmag = np.sqrt(bx ** 2 + by ** 2 + bz ** 2)
    bhx, bhy, bhz = bx / Bmag, by / Bmag, bz / Bmag

    # reference direction: x-hat, unless b is nearly parallel to x, then use y-hat
    use_y = np.abs(bhx) > 0.9
    refx = np.where(use_y, 0.0, 1.0)
    refy = np.where(use_y, 1.0, 0.0)
    refz = np.zeros_like(bhx)

    e1x = bhy * refz - bhz * refy
    e1y = bhz * refx - bhx * refz
    e1z = bhx * refy - bhy * refx
    e1mag = np.sqrt(e1x ** 2 + e1y ** 2 + e1z ** 2)
    e1x, e1y, e1z = e1x / e1mag, e1y / e1mag, e1z / e1mag

    e2x = bhy * e1z - bhz * e1y
    e2y = bhz * e1x - bhx * e1z
    e2z = bhx * e1y - bhy * e1x

    return (bhx, bhy, bhz), (e1x, e1y, e1z), (e2x, e2y, e2z)

def add_field_aligned_velocities(df):
    (bhx, bhy, bhz), (e1x, e1y, e1z), (e2x, e2y, e2z) = perpendicular_basis(
        df['bx'].to_numpy(dtype=float), df['by'].to_numpy(dtype=float), df['bz'].to_numpy(dtype=float)
    )
    u = df['u'].to_numpy(dtype=float)
    v = df['v'].to_numpy(dtype=float)
    w = df['w'].to_numpy(dtype=float)

    df['v_par'] = u * bhx + v * bhy + w * bhz
    df['v_perp1'] = u * e1x + v * e1y + w * e1z
    df['v_perp2'] = u * e2x + v * e2y + w * e2z

for df in region_dfs.values():
    add_field_aligned_velocities(df)

In [ ]:
def fit_drifting_maxwellian_components(df, components):
    weights = df['wei'].to_numpy(dtype=float)
    fit = {}
    for component in components:
        values = df[component].to_numpy(dtype=float)
        fit[component] = weighted_mean_std(values, weights)
    return fit

fab_fits = {idx: fit_drifting_maxwellian_components(df, ['v_par', 'v_perp1', 'v_perp2']) for idx, df in region_dfs.items()}
fab_fits

In [ ]:
# Plot each species' field-aligned velocity distributions with the Maxwellian fits overlaid

fig, axes = plt.subplots(2, 3, figsize=(8, 4), constrained_layout=True)

fab_components = ['v_par', 'v_perp1', 'v_perp2']
fab_labels = [r'$v_\parallel$', r'$v_{\perp,1}$', r'$v_{\perp,2}$']

for row, idx in enumerate(species_names):
    df = region_dfs[idx]
    weights = df['wei'].to_numpy(dtype=float)

    for col, (component, label) in enumerate(zip(fab_components, fab_labels)):
        ax = axes[row, col]
        values = df[component].to_numpy(dtype=float)

        hist, bin_edges = np.histogram(values, bins=60, weights=weights, density=True)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        mu, sigma = fab_fits[idx][component]
        v_fit = np.linspace(bin_edges[0], bin_edges[-1], 300)

        ax.bar(bin_centers, hist, width=np.diff(bin_edges), color='steelblue', alpha=0.6, label='simulation')
        ax.plot(v_fit, gaussian(v_fit, mu, sigma), 'r--', lw=2, label=fr'fit: $\mu$={mu:.2e}, $\sigma$={sigma:.2e}')

        ax.set_xlabel(label)
        if col == 0:
            ax.set_ylabel('density')
        ax.set_title(f'{species_names[idx]}: {label}')
        ax.legend(fontsize=4)
        ax.grid(alpha=0.3)

fig.suptitle(f"Field-aligned drifting Maxwellian fits, findex={findex}, step={chosen_step}")
plt.show()

In [ ]:
# Parallel/perpendicular temperature from the thermal spread along and across the local B field
# T_par/(m c^2) = sigma_par^2; T_perp/(m c^2) = mean of the two perpendicular variances

temperatures_par = {}
temperatures_perp = {}

for idx in species_names:
    sigma_par = fab_fits[idx]['v_par'][1]
    sigma_perp1 = fab_fits[idx]['v_perp1'][1]
    sigma_perp2 = fab_fits[idx]['v_perp2'][1]

    theta_par = sigma_par ** 2
    theta_perp = np.mean([sigma_perp1 ** 2, sigma_perp2 ** 2])

    temperatures_par[idx] = masses[idx] * theta_par
    temperatures_perp[idx] = masses[idx] * theta_perp

    print(f"{species_names[idx]}: T_par/(m_e c^2) = {temperatures_par[idx]:.4e}, "
          f"T_perp/(m_e c^2) = {temperatures_perp[idx]:.4e}, T_perp/T_par = {theta_perp / theta_par:.3f}")

T_ratio_par = temperatures_par[2] / temperatures_par[1]
T_ratio_perp = temperatures_perp[2] / temperatures_perp[1]
print(f"\nT_i/T_e (parallel) = {T_ratio_par:.3f}")
print(f"T_i/T_e (perpendicular) = {T_ratio_perp:.3f}")

In [ ]:
# T_parallel / T_perp in the region

fig, axes = plt.subplots(1, 2, figsize=(9, 5), sharey=True)

axes[0].bar(['electron', 'ion'], [temperatures_par[1], temperatures_par[2]], color=['steelblue', 'indianred'])
axes[0].set_title(fr'$T_\parallel$: $T_i/T_e$ = {T_ratio_par:.2f}')
axes[0].set_ylabel('$T / (m_e c^2)$')
axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(['electron', 'ion'], [temperatures_perp[1], temperatures_perp[2]], color=['steelblue', 'indianred'])
axes[1].set_title(fr'$T_\perp$: $T_i/T_e$ = {T_ratio_perp:.2f}')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Ion energy spectrum, lab frame vs. flow frame, log-log to look for a power-law tail

df_ion = region_dfs[2]
weights_ion = df_ion['wei'].to_numpy(dtype=float)
m_ion = masses[2]

u_ion = df_ion['u'].to_numpy(dtype=float)
v_ion = df_ion['v'].to_numpy(dtype=float)
w_ion = df_ion['w'].to_numpy(dtype=float)

mu_u, mu_v, mu_w = bulk_velocity[2]

energy_lab = 0.5 * m_ion * (u_ion ** 2 + v_ion ** 2 + w_ion ** 2)
energy_flow = 0.5 * m_ion * ((u_ion - mu_u) ** 2 + (v_ion - mu_v) ** 2 + (w_ion - mu_w) ** 2)

def weighted_log_spectrum(energy, weights, num_bins=40):
    positive = energy > 0
    bins = np.logspace(np.log10(energy[positive].min()), np.log10(energy[positive].max()), num_bins + 1)
    hist, bin_edges = np.histogram(energy, bins=bins, weights=weights, density=True)
    bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # geometric mean, natural for log bins
    return bin_centers, hist

bin_centers_lab, hist_lab = weighted_log_spectrum(energy_lab, weights_ion)
bin_centers_flow, hist_flow = weighted_log_spectrum(energy_flow, weights_ion)

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(bin_centers_lab, hist_lab, 'o-', ms=4, label='lab frame', color='indianred')
ax.loglog(bin_centers_flow, hist_flow, 's-', ms=4, label='flow frame', color='darkorange')

ax.set_xlabel(r'$E = \frac{1}{2} m_i v^2$  $[m_e c^2]$')
ax.set_ylabel(r'$dN/dE$')
ax.set_title(f'Ion energy spectrum, findex={findex}, step={chosen_step}')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(run.sigma_eff)

In [ ]:
# Electron and ion energy spectra in each species' own flow frame (polished)

def flow_frame_energy_spectrum(idx, num_bins=20):
    df = region_dfs[idx]
    weights = df['wei'].to_numpy(dtype=float)
    mass = masses[idx]

    velocity = df[['u', 'v', 'w']].to_numpy(dtype=float)
    drift = np.asarray(bulk_velocity[idx])
    energy = 0.5 * mass * np.sum((velocity - drift) ** 2, axis=1)

    positive = energy > 0
    bins = np.logspace(
        np.log10(energy[positive].min()),
        np.log10(energy[positive].max()),
        num_bins + 1
    )
    hist, edges = np.histogram(energy, bins=bins, weights=weights, density=True)
    hist /= np.sum(hist * np.diff(edges))  # normalize so integral of f(E) dE is 1
    centers = np.sqrt(edges[:-1] * edges[1:])
    return centers, hist

def maxwellian_energy_pdf(E, T):
    # isotropic non-relativistic Maxwell-Boltzmann energy distribution; T in units of m_e c^2,
    # matching the isotropic fit temperature from `temperatures` (see fit cell above)
    return 2 * np.sqrt(E / np.pi) * T ** -1.5 * np.exp(-E / T)

bin_centers_electron, hist_electron = flow_frame_energy_spectrum(1)
bin_centers_ion, hist_ion = flow_frame_energy_spectrum(2)

colors = {1: 'steelblue', 2: 'indianred'}

with plt.rc_context({'font.size': 16}):
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.loglog(
        bin_centers_electron, hist_electron,
        'o-', ms=4, label=fr'electron, $T_e/(m_i v_A^2)$ = {temperatures[1] / (run.sigma_eff * run.mime):.2f}', color=colors[1]
    )
    ax.loglog(
        bin_centers_ion, hist_ion,
        's-', ms=4, label=fr'ion, $T_i/(m_i v_A^2)$ = {temperatures[2] / (run.sigma_eff * run.mime):.2f}', color=colors[2]
    )

    E_fit_electron = np.logspace(np.log10(bin_centers_electron.min()), np.log10(bin_centers_electron.max()), 200)
    E_fit_ion = np.logspace(np.log10(bin_centers_ion.min()), np.log10(bin_centers_ion.max()), 200)

    ax.loglog(E_fit_electron, maxwellian_energy_pdf(E_fit_electron, temperatures[1]), '--', color=colors[1])
    ax.loglog(E_fit_ion, maxwellian_energy_pdf(E_fit_ion, temperatures[2]), '--', color=colors[2])

    ax.set_xlim(1e-3, 0.33)

    ax.set_xlabel(r'$E_{\mathrm{flow\ frame}} = \frac{1}{2}m_s|\mathbf{v}-\mathbf{v}_{\mathrm{bulk},s}|^2$')
    ax.set_ylabel(r'$f(E)$')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    plt.tight_layout()

    title = f'Electron and ion flow-frame energy spectra, findex={findex}, step={chosen_step}'
    print(title)
    metadata = {'title': title}
    save_polished_fig('ion_electron_flow_frame_spectra.png', metadata=metadata)

    plt.show()